In [1]:
import os
os.environ["DISABLE_TRANSFORMERS_FLASH_ATTN"] = "1"

# Load data

In [2]:
import pandas as pd
df = df = pd.read_csv("quality_data.csv")
print(df.head())

                     narration  mode    type    category  subcategory
0  interest from fixed deposit  NEFT  Credit  investment  fd_interest
1           FD interest payout  NEFT  Credit  investment  fd_interest
2       Fixed deposit interest  NEFT  Credit  investment  fd_interest
3       bank interest received  NEFT  Credit  investment  fd_interest
4         FD maturity interest  NEFT  Credit  investment  fd_interest


# Create input_text

In [3]:
df["input_text"] = (
    "narration: " + df["narration"].astype(str) +
    " | mode: " + df["mode"].astype(str) +
    " | type: " + df["type"].astype(str)
)

## Lowercase

In [4]:
df["input_text"] = df["input_text"].str.lower()

# Create label (Ground Truth)

In [5]:
df["label"] = df["category"] + "_" + df["subcategory"]

In [6]:
print(df[["input_text", "label"]].head())

                                          input_text                   label
0  narration: interest from fixed deposit | mode:...  investment_fd_interest
1  narration: fd interest payout | mode: neft | t...  investment_fd_interest
2  narration: fixed deposit interest | mode: neft...  investment_fd_interest
3  narration: bank interest received | mode: neft...  investment_fd_interest
4  narration: fd maturity interest | mode: neft |...  investment_fd_interest


In [7]:
from datasets import Dataset

df["input_text"] = (
    df["narration"] + " " + df["mode"] + " " + df["type"]
)

df["input_text"] = df["input_text"].str.lower()

dataset = Dataset.from_pandas(df[["input_text", "label"]])

In [8]:
labels = list(df["label"].unique())
label2id = {l: i for i, l in enumerate(labels)}
id2label = {i: l for l, i in label2id.items()}

def encode(example):
    example["label"] = label2id[example["label"]]
    return example

dataset = dataset.map(encode)

Map:   0%|          | 0/91 [00:00<?, ? examples/s]

In [9]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")

def tokenize(example):
    return tokenizer(example["input_text"], truncation=True, padding="max_length")

dataset = dataset.map(tokenize)


A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.2.6 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "c:\Users\ACER\AppData\Local\Programs\Python\Python310\lib\runpy.py", line 196, in _run_module_as_main
    return _run_code(code, main_globals, None,
  File "c:\Users\ACER\AppData\Local\Programs\Python\Python310\lib\runpy.py", line 86, in _run_code
    exec(code, run_globals)
  File "c:\Users\ACER\AppData\Local\Programs\Python\Python310\lib\site-packages\ipykernel_launcher.py", line 18, in <module>
    app.launch_new_instance()
  File "c:\Users\ACER\AppData\Local\Programs\Python\Python310\lib\site-packages\traitlets

Map:   0%|          | 0/91 [00:00<?, ? examples/s]

In [10]:
dataset = dataset.train_test_split(test_size=0.2)

train_dataset = dataset["train"]
test_dataset = dataset["test"]

In [11]:
from transformers import AutoModelForSequenceClassification, Trainer, TrainingArguments

model = AutoModelForSequenceClassification.from_pretrained(
    "distilbert-base-uncased",
    num_labels=len(labels),
    id2label=id2label,
    label2id=label2id
)

training_args = TrainingArguments(
    output_dir="./results",
    per_device_train_batch_size=4,
    num_train_epochs=6,
    logging_steps=10,
    save_strategy="no",
    report_to="none"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset
)

trainer.train()

c:\Users\ACER\AppData\Local\Programs\Python\Python310\lib\site-packages\google\api_core\_python_version_support.py:263: FutureWarning: You are using a Python version (3.10.6) which Google will stop supporting in new releases of google.api_core once it reaches its end of life (2026-10-04). Please upgrade to the latest Python version, or at least Python 3.11, to continue receiving updates for google.api_core past that date.
  warnings.warn(message, FutureWarning)
Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


  0%|          | 0/108 [00:00<?, ?it/s]

{'loss': 2.4163, 'grad_norm': 4.2809600830078125, 'learning_rate': 4.5370370370370374e-05, 'epoch': 0.56}
{'loss': 2.244, 'grad_norm': 5.203774929046631, 'learning_rate': 4.074074074074074e-05, 'epoch': 1.11}
{'loss': 2.0613, 'grad_norm': 6.354163646697998, 'learning_rate': 3.611111111111111e-05, 'epoch': 1.67}
{'loss': 1.8498, 'grad_norm': 9.472647666931152, 'learning_rate': 3.148148148148148e-05, 'epoch': 2.22}
{'loss': 1.6022, 'grad_norm': 7.846771240234375, 'learning_rate': 2.6851851851851855e-05, 'epoch': 2.78}
{'loss': 1.4591, 'grad_norm': 6.35092306137085, 'learning_rate': 2.2222222222222223e-05, 'epoch': 3.33}
{'loss': 1.44, 'grad_norm': 7.141161918640137, 'learning_rate': 1.7592592592592595e-05, 'epoch': 3.89}
{'loss': 1.2027, 'grad_norm': 7.333642482757568, 'learning_rate': 1.2962962962962962e-05, 'epoch': 4.44}
{'loss': 1.1679, 'grad_norm': 6.191513538360596, 'learning_rate': 8.333333333333334e-06, 'epoch': 5.0}
{'loss': 1.0349, 'grad_norm': 4.853950023651123, 'learning_rate

TrainOutput(global_step=108, training_loss=1.6071552612163402, metrics={'train_runtime': 1219.2928, 'train_samples_per_second': 0.354, 'train_steps_per_second': 0.089, 'total_flos': 57235101106176.0, 'train_loss': 1.6071552612163402, 'epoch': 6.0})

In [12]:
import torch
import torch.nn.functional as F

def predict(text):
    inputs = tokenizer(text, return_tensors="pt", truncation=True)
    outputs = model(**inputs)

    probs = F.softmax(outputs.logits, dim=1)

    confidence = torch.max(probs).item()
    pred_id = torch.argmax(probs).item()

    label = id2label[pred_id]
    category, subcategory = label.split("_", 1)

    return category, subcategory, round(confidence, 2)

def confidence_level(conf):
    if conf >= 0.65:
        return "High"
    elif conf >= 0.4:
        return "Medium"
    else:
        return "Low"

In [13]:
results = []

for i in range(len(df)):
    narration = df["narration"].iloc[i]
    mode = df["mode"].iloc[i]
    txn_type = df["type"].iloc[i]

    # same input used for model
    text = df["input_text"].iloc[i]

    pred_category, pred_subcategory, confidence = predict(text)

    results.append({
        "narration": narration,
        "mode": mode,
        "type": txn_type,
        "category": df["category"].iloc[i],
        "subcategory": df["subcategory"].iloc[i],
        "predicted_category": pred_category,
        "predicted_subcategory": pred_subcategory,
        "confidence": confidence,
        "confidence_percent": f"{round(confidence*100,1)}%",
        "confidence_level": confidence_level(confidence),
        "correct": (df["category"].iloc[i] == pred_category) and 
           (df["subcategory"].iloc[i] == pred_subcategory)
    })

result_df = pd.DataFrame(results)

print(result_df.to_string())

                        narration        mode    type    category  subcategory predicted_category predicted_subcategory  confidence confidence_percent confidence_level  correct
0     interest from fixed deposit        NEFT  Credit  investment  fd_interest         investment           fd_interest        0.35              35.0%              Low     True
1              FD interest payout        NEFT  Credit  investment  fd_interest         investment           fd_interest        0.38              38.0%              Low     True
2          Fixed deposit interest        NEFT  Credit  investment  fd_interest         investment           fd_interest        0.36              36.0%              Low     True
3          bank interest received        NEFT  Credit  investment  fd_interest         investment           fd_interest        0.36              36.0%              Low     True
4            FD maturity interest        NEFT  Credit  investment  fd_interest         investment           fd_inte

## | Metric | Meaning      | Use here               |
### | ------ | ------------ | ---------------------- |
### | F1     | balance      | good baseline          |
### | F2     | recall heavy | better for your system |


In [14]:
y_true = result_df["category"] + "_" + result_df["subcategory"]
y_pred = result_df["predicted_category"] + "_" + result_df["predicted_subcategory"]

In [15]:
from sklearn.metrics import f1_score, fbeta_score

f1 = f1_score(y_true, y_pred, average='weighted')
f2 = fbeta_score(y_true, y_pred, beta=2, average='weighted')

print("Overall Metrics:")
print(f"F1 Score: {round(f1,3)}")
print(f"F2 Score: {round(f2,3)}")

Overall Metrics:
F1 Score: 0.782
F2 Score: 0.785


In [16]:
from sklearn.metrics import classification_report
import pandas as pd

report = classification_report(y_true, y_pred, output_dict=True)

report_df = pd.DataFrame(report).transpose()

print(report_df)

                        precision    recall  f1-score    support
expense_bills            1.000000  0.875000  0.933333   8.000000
expense_food             1.000000  0.125000  0.222222   8.000000
expense_health           1.000000  1.000000  1.000000   7.000000
expense_insurance        1.000000  1.000000  1.000000   8.000000
expense_loan             1.000000  1.000000  1.000000   8.000000
expense_shopping         0.500000  1.000000  0.666667   8.000000
expense_transport        0.900000  0.900000  0.900000  10.000000
income_salary            1.000000  0.500000  0.666667   8.000000
investment_fd_booking    1.000000  1.000000  1.000000   8.000000
investment_fd_interest   0.526316  1.000000  0.689655  10.000000
investment_stocks        1.000000  0.375000  0.545455   8.000000
accuracy                 0.802198  0.802198  0.802198   0.802198
macro avg                0.902392  0.797727  0.784000  91.000000
weighted avg             0.893002  0.802198  0.782102  91.000000


In [17]:
from sklearn.metrics import fbeta_score

labels = list(set(y_true))

f2_scores = {}

for label in labels:
    y_true_bin = [1 if y == label else 0 for y in y_true]
    y_pred_bin = [1 if y == label else 0 for y in y_pred]

    f2_scores[label] = fbeta_score(y_true_bin, y_pred_bin, beta=2)

report_df["f2_score"] = report_df.index.map(f2_scores)

In [18]:
report_df = report_df.round(3)
print(report_df)

                        precision  recall  f1-score  support  f2_score
expense_bills               1.000   0.875     0.933    8.000     0.897
expense_food                1.000   0.125     0.222    8.000     0.152
expense_health              1.000   1.000     1.000    7.000     1.000
expense_insurance           1.000   1.000     1.000    8.000     1.000
expense_loan                1.000   1.000     1.000    8.000     1.000
expense_shopping            0.500   1.000     0.667    8.000     0.833
expense_transport           0.900   0.900     0.900   10.000     0.900
income_salary               1.000   0.500     0.667    8.000     0.556
investment_fd_booking       1.000   1.000     1.000    8.000     1.000
investment_fd_interest      0.526   1.000     0.690   10.000     0.847
investment_stocks           1.000   0.375     0.545    8.000     0.429
accuracy                    0.802   0.802     0.802    0.802       NaN
macro avg                   0.902   0.798     0.784   91.000       NaN
weight

In [19]:
print("\nBest Performing Category:")
print(report_df["f1-score"].idxmax(), report_df["f1-score"].max())

print("\nWorst Performing Category:")
print(report_df["f1-score"].idxmin(), report_df["f1-score"].min())


Best Performing Category:
expense_health 1.0

Worst Performing Category:
expense_food 0.222
